In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 6 - WEEK 11 BAYESIAN OPTIMISATION
# Run from inside week11/
# ============================================================
#
# Strategy:
# - Refit ARD Matern GP including Week 10.
# - Check Week 10 realised calibration.
# - Centre search on the ACTUAL incumbent.
# - Use a conservative empirical trust region.
# - Compare EI, posterior mean and UCB.
# - Do not automatically expand if boundary is hit.
# ============================================================


# ------------------------------------------------------------
# 1. Load cumulative Week 11 data
# ------------------------------------------------------------

X = np.load("function6/initial_inputs.npy")
Y = np.load("function6/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 10 calibration check
# ------------------------------------------------------------
#
# Week 10 selected:
# [0.44566108, 0.40694577, 0.58867390,
#  0.70237278, 0.12162548]
#
# Prior prediction:
# mean ≈ -0.223154
# std  ≈ 0.038525
#
# Actual:
# -0.2629443056177093
# ------------------------------------------------------------

week10_pred_mean = -0.2231541596877067
week10_pred_std = 0.038525287827434716
week10_actual = -0.2629443056177093

week10_error = (
    week10_actual
    - week10_pred_mean
)

week10_z_error = (
    week10_error
    / week10_pred_std
)

print("\n================================")
print("WEEK 10 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week10_pred_mean)
print("Predicted std :", week10_pred_std)
print("Actual        :", week10_actual)

print("\nPrediction error:")
print(week10_error)

print("\nError / predicted std:")
print(week10_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(5) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask] - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

empirical_cap = min(
    1.25 * nearest_distance,
    0.06
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print("Nearest observed point:")
print(nearest_distance)

print("\nEmpirical cap:")
print(empirical_cap)


# ------------------------------------------------------------
# 6. Conservative ARD trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.18 * lengthscales,
    0.02,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x + trust_half_width
)

print("\n================================")
print("WEEK 11 TRUST REGION")
print("================================")

print("Centre:")
print(best_x)

print("\nHalf-widths:")
print(trust_half_width)

print("\nLower:")
print(lower)

print("\nUpper:")
print(upper)


# ------------------------------------------------------------
# 7. Dense trust-region candidates
# ------------------------------------------------------------

rng = np.random.default_rng(42)

candidates = rng.uniform(
    lower,
    upper,
    size=(400000, 5)
)

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 8. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 9. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 10. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 11. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 12. Distance from incumbent
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 13. Boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(len(x)):

        if abs(
            x[j] - lower[j]
        ) <= tol:

            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(
            x[j] - upper[j]
        ) <= tol:

            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(status)


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )

DATA
X shape: (30, 5)
Y shape: (30,)

Current best:
[0.442929 0.409333 0.63183  0.7398   0.129381] -> -0.19517557780237

Y range:
min = -2.5711696316081234
max = -0.19517557780237
std = 0.6599453906388283

WEEK 10 CALIBRATION CHECK
Predicted mean: -0.2231541596877067
Predicted std : 0.038525287827434716
Actual        : -0.2629443056177093

Prediction error:
-0.03979014593000263

Error / predicted std:
-1.032831892346491

GP FIT

Fitted kernel:
1.38**2 * Matern(length_scale=[0.878, 1.03, 1.45, 1.01, 1.13], nu=2.5) + WhiteKernel(noise_level=0.00296)

ARD lengthscales:
[0.87761467 1.03243031 1.44932809 1.00964659 1.13289344]

Normalised inverse-lengthscale sensitivity:
[0.24393364 0.20735515 0.14770965 0.21203433 0.18896723]

EMPIRICAL LOCAL SCALE
Nearest observed point:
0.0453996159344988

Empirical cap:
0.056749519918123506

WEEK 11 TRUST REGION
Centre:
[0.442929 0.409333 0.63183  0.7398   0.129381]

Half-widths:
[0.05674952 0.05674952 0.05674952 0.05674952 0.05674952]

Lower:
[0.386179

In [2]:
# ============================================================
# FINAL FUNCTION 6 - WEEK 11 SELECTION
# ============================================================

beta = 0.25

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week11_candidate = candidates[final_idx]

print("Week 11 Function 6 candidate:")
print(week11_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week11_candidate - best_x
    )
)

portal = "-".join(
    f"{x:.6f}"
    for x in week11_candidate
)

print("\nPortal format:")
print(portal)

Week 11 Function 6 candidate:
[0.44463181 0.41069563 0.58658281 0.71195984 0.11803155]

Predicted mean:
-0.23128728702383516

Predicted std:
0.03895291141835071

Distance from current best:
0.054368645029322094

Portal format:
0.444632-0.410696-0.586583-0.711960-0.118032
